In [1]:
"""
📌 Medical RAG System with FAISS + BioBERT + Optional Cross-Encoder Reranking + Flan-T5
Evaluation + Interactive GUI
Optimized for readable Evaluation outputs and complete RAG answers
"""

# ====================================================
# 1️⃣ Install dependencies
# ====================================================
!pip install -q transformers torch faiss-cpu tqdm datasets scikit-learn ipywidgets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 46.9 MB/s eta 0:00:00


In [2]:
# ====================================================
# 2️⃣ Imports
# ====================================================
import re, csv, numpy as np
from pathlib import Path
from tqdm import tqdm
import torch, faiss
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForSeq2SeqLM,
    AutoModelForSequenceClassification
)
import ipywidgets as widgets
from IPython.display import display, clear_output


In [3]:
# ====================================================
# 3️⃣ Global Settings
# ====================================================
BASE_DIR = Path("/content")
INDEX_DIR = BASE_DIR / "faiss_index"
INDEX_DIR.mkdir(exist_ok=True)

INDEX_PATH = INDEX_DIR / "pubmed.index"
META_PATH = INDEX_DIR / "metadata.csv"

EMBED_DIM = 768
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MAX_CE_LENGTH = 512
MAX_LLM_INPUT_TOKENS = 450
MAX_LLM_OUTPUT_TOKENS = 512

In [4]:
# ====================================================
# 4️⃣ BioBERT Encoder
# ====================================================
print("🔹 Loading BioBERT encoder...")
biobert_tokenizer = AutoTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.1")
biobert_model = AutoModel.from_pretrained("dmis-lab/biobert-base-cased-v1.1").to(DEVICE)
biobert_model.eval()

def encode_text(text: str):
    tokens = biobert_tokenizer(
        text, return_tensors="pt", truncation=True, padding=True, max_length=512
    ).to(DEVICE)
    with torch.no_grad():
        outputs = biobert_model(**tokens)
        embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings.cpu().numpy().astype("float32")


🔹 Loading BioBERT encoder...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

In [5]:
# -*- coding: utf-8 -*-





# ====================================================
# 5️⃣ Text Preprocessing
# ====================================================
ABBREVIATIONS = {
    "mi":"myocardial infarction","bp":"blood pressure","dm":"diabetes mellitus",
    "copd":"chronic obstructive pulmonary disease","cvd":"cardiovascular disease",
    "htn":"hypertension"
}

def clean_text(text:str):
    text = re.sub(r"\[[0-9]+\]", " ", text)
    text = re.sub(r"\([0-9]{4}\)", " ", text)
    text = re.sub(r"[^a-zA-Z0-9.,;:%()\- ]", " ", text)
    return re.sub(r"\s+"," ", text).strip()

def expand_abbreviations(text:str):
    return " ".join(ABBREVIATIONS.get(w.lower(), w) for w in text.split())

def sentence_split(text:str): return re.split(r'(?<=[.!?])\s+', text)

def chunk_sentences(text:str, max_words=180):
    sentences = sentence_split(text)
    chunks, current, count = [], [], 0
    for sent in sentences:
        words = sent.split()
        if count + len(words) > max_words:
            chunks.append(" ".join(current)); current, count = [], 0
        current.append(sent)
        count += len(words)
    if current: chunks.append(" ".join(current))
    return chunks

def preprocess_document(text:str):
    return chunk_sentences(expand_abbreviations(clean_text(text)))

def preprocess_query(text:str): return expand_abbreviations(clean_text(text))
def is_valid_chunk(text:str, min_words=25): return len(text.split())>=min_words

In [6]:


# ====================================================
# 6️⃣ Load Dataset & Build FAISS Index
# ====================================================
print("📥 Loading PubMed dataset...")
dataset = load_dataset("slinusc/PubMedAbstractsSubset", split="train[:1000]")
print("✅ Dataset loaded")

index = faiss.IndexFlatL2(EMBED_DIM)
metadata = []

for item in tqdm(dataset):
    pmid = int(item["PMID"])
    raw_text = f"{item['title']} {item['abstract']}".strip()
    chunks = preprocess_document(raw_text)
    for cid, chunk in enumerate(chunks):
        if not is_valid_chunk(chunk): continue
        emb = encode_text(chunk)
        index.add(emb)
        metadata.append([pmid, cid, chunk])

faiss.write_index(index, str(INDEX_PATH))
with open(META_PATH,"w",newline="",encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["PMID","ChunkID","Text"])
    writer.writerows(metadata)
print("✅ FAISS index built")

📥 Loading PubMed dataset...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/24 [00:00<?, ?it/s]

pubmed_chunk_0.jsonl:   0%|          | 0.00/111M [00:00<?, ?B/s]

pubmed_chunk_1.jsonl:   0%|          | 0.00/120M [00:00<?, ?B/s]

...

pubmed_chunk_9.jsonl:   0%|          | 0.00/141M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2392165 [00:00<?, ? examples/s]

✅ Dataset loaded


100%|██████████| 1000/1000 [00:21<00:00, 47.60it/s]

✅ FAISS index built


In [7]:
# ====================================================
# 7️⃣ FAISS Retrieval
# ====================================================
def retrieve(query_emb,k=10): _, idxs = index.search(query_emb,k); return idxs[0]
def retrieve_pmids(query_text,k=10): return [metadata[i][0] for i in retrieve(encode_text(preprocess_query(query_text)),k)]


In [8]:
# ====================================================
# 8️⃣ Cross-Encoder
# ====================================================
print("🔹 Loading Cross-Encoder...")
ce_tokenizer = AutoTokenizer.from_pretrained("cross-encoder/ms-marco-MiniLM-L-6-v2")
ce_model = AutoModelForSequenceClassification.from_pretrained("cross-encoder/ms-marco-MiniLM-L-6-v2").to(DEVICE)
ce_model.eval()

def cross_encoder_rerank(query, candidate_indices, top_m=5):
    pairs = [(query,metadata[i][2]) for i in candidate_indices]
    inputs = ce_tokenizer(pairs, padding=True, truncation=True, max_length=MAX_CE_LENGTH, return_tensors="pt").to(DEVICE)
    with torch.no_grad(): scores = ce_model(**inputs).logits.squeeze(-1)
    scores = scores.cpu().numpy()
    ranked = np.argsort(scores)[::-1][:top_m]
    return [candidate_indices[i] for i in ranked]

def retrieve_with_rerank(query_text,k=20,top_m=5):
    emb = encode_text(preprocess_query(query_text))
    initial_idxs = retrieve(emb,k)
    return [metadata[i][0] for i in cross_encoder_rerank(query_text, initial_idxs, top_m)]


🔹 Loading Cross-Encoder...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

In [9]:




# ====================================================
# 9️⃣ Evaluation Metrics
# ====================================================
def precision_at_k(retrieved,relevant,k): return sum(1 for r in retrieved[:k] if r==relevant)/k
def recall_at_k(retrieved,relevant,k): return 1.0 if relevant in retrieved[:k] else 0.0
def mrr(retrieved,relevant):
    for i,r in enumerate(retrieved):
        if r==relevant: return 1/(i+1)
    return 0.0
def ndcg_at_k(retrieved,relevant,k):
    for i,r in enumerate(retrieved[:k]):
        if r==relevant: return 1/np.log2(i+2)
    return 0.0

def build_eval_queries(dataset,n=20):
    return [{"query":dataset[i]["title"],"relevant_pmid":int(dataset[i]["PMID"])} for i in range(n)]

eval_queries = build_eval_queries(dataset)

def evaluate(eval_queries,k=10,use_rerank=False):
    P,R,M,N = [],[],[],[]
    for q in eval_queries:
        retrieved = retrieve_with_rerank(q["query"],k=k,top_m=k) if use_rerank else retrieve_pmids(q["query"],k)
        rel = q["relevant_pmid"]
        P.append(precision_at_k(retrieved,rel,k))
        R.append(recall_at_k(retrieved,rel,k))
        M.append(mrr(retrieved,rel))
        N.append(ndcg_at_k(retrieved,rel,k))
    # تبدیل به float معمولی و بازگشت مرتب
    return {k: float(v) for k,v in zip(["Precision@10","Recall@10","MRR","NDCG@10"], [np.mean(P),np.mean(R),np.mean(M),np.mean(N)])}

metrics_faiss = evaluate(eval_queries,k=10,use_rerank=False)
metrics_rerank = evaluate(eval_queries,k=10,use_rerank=True)

def print_metrics(metrics,title="Metrics"):
    print(f"\n📊 {title}")
    for k,v in metrics.items():
        print(f"{k}: {v:.4f}")

print_metrics(metrics_faiss,"FAISS-only")
print_metrics(metrics_rerank,"FAISS + Cross-Encoder")



📊 FAISS-only
Precision@10: 0.1250
Recall@10: 0.9500
MRR: 0.8446
NDCG@10: 0.8697

📊 FAISS + Cross-Encoder
Precision@10: 0.1250
Recall@10: 0.9500
MRR: 0.9500
NDCG@10: 0.9500


In [10]:

# ====================================================
# 1️⃣1️⃣ Load Flan-T5
# ====================================================
llm_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
llm_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base").to(DEVICE)

def call_llm(prompt):
    inputs = llm_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_CE_LENGTH).to(DEVICE)
    with torch.no_grad():
        output = llm_model.generate(**inputs, max_new_tokens=MAX_LLM_OUTPUT_TOKENS)
    return llm_tokenizer.decode(output[0], skip_special_tokens=True)


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [11]:
# ====================================================
# 1️⃣2️⃣ RAG Demo
# ====================================================
def search_and_answer(query,top_k_chunks=5):
    emb = encode_text(preprocess_query(query))
    idxs = retrieve(emb,k=top_k_chunks*2)
    idxs_rerank = cross_encoder_rerank(query,idxs,top_m=top_k_chunks)
    contexts = [metadata[i][2] for i in idxs_rerank]
    context_text = ' '.join(contexts).split()[:MAX_LLM_INPUT_TOKENS]
    context_text = ' '.join(context_text)
    prompt = f"Using the following context, answer the medical question in detail and in complete sentences.\n\nContext:\n{context_text}\n\nQuestion:\n{query}\n\nAnswer:"
    return call_llm(prompt)

print("🧠 Sample RAG Answer:")
print(search_and_answer("What cellular changes occur during myocardial infarction?"))


🧠 Sample RAG Answer:
preventing myocardial cellular hypoxia.


In [12]:


# ====================================================
# 1️⃣3️⃣ Interactive GUI
# ====================================================
query_box = widgets.Textarea(value='', placeholder='سوال پزشکی خود را اینجا بنویسید...', description='سوال:', layout=widgets.Layout(width='600px',height='80px'))
mode_selector = widgets.RadioButtons(options=['FAISS-only','FAISS + Cross-Encoder'], value='FAISS + Cross-Encoder', description='Mode:', layout=widgets.Layout(width='250px'))
submit_button = widgets.Button(description='پرسش و دریافت پاسخ', button_style='success', layout=widgets.Layout(width='250px'))
output = widgets.Output(layout=widgets.Layout(width='700px'))

def on_submit_clicked(b):
    with output:
        clear_output()
        query = query_box.value.strip()
        if not query: print("❌ لطفاً یک سوال وارد کنید."); return
        print("⏳ در حال پردازش...")
        emb = encode_text(preprocess_query(query))
        if mode_selector.value=='FAISS-only':
            idxs = retrieve(emb,k=5)
            contexts = [metadata[i][2] for i in idxs]
        else:
            idxs = retrieve(emb,k=10)
            idxs_rerank = cross_encoder_rerank(query,idxs,top_m=5)
            contexts = [metadata[i][2] for i in idxs_rerank]
        context_text = ' '.join(contexts).split()[:MAX_LLM_INPUT_TOKENS]
        context_text = ' '.join(context_text)
        prompt = f"Using the following context, answer the medical question in detail and in complete sentences.\n\nContext:\n{context_text}\n\nQuestion:\n{query}\n\nAnswer:"
        answer = call_llm(prompt)
        print("🧠 پاسخ:")
        print(answer)

submit_button.on_click(on_submit_clicked)
display(query_box); display(mode_selector); display(submit_button); display(output)


Textarea(value='', description='سوال:', layout=Layout(height='80px', width='600px'), placeholder='سوال پزشکی خ…

RadioButtons(description='Mode:', index=1, layout=Layout(width='250px'), options=('FAISS-only', 'FAISS + Cross…

Button(button_style='success', description='پرسش و دریافت پاسخ', layout=Layout(width='250px'), style=ButtonSty…

Output(layout=Layout(width='700px'))